# Soup take-home: LoRA → merge → 4-bit → SHIP / DON'T SHIP (Qwen2.5-1.5B-Instruct, Russian, Colab T4)

Run top to bottom on a **free Colab T4** (Runtime → Change runtime type → T4 GPU). The kit lives in `/content/soup-task`
(clone or upload, see `README.md`). Outputs: `logs/` (raw, timestamped) and `outputs/` (JSON, tables, plots), all zipped at the end.

| § | Step | T4 time (expected, not measured) |
|---|---|---|
| 0 | Setup, versions, logging, nvidia-smi sampler, Soup CLI discovery | 4–7 min |
| 1 | Data: download, filter, dedupe, leakage check, length stats; template + loss-mask check | 2–4 min |
| 2 | Memory budget **before training** (+ `soup profile`, CUDA context calibration) | ~1 min |
| 3 | Soup data checks ("data doctor") + pre-flight (`soup doctor`, `soup plan`, `soup train --dry-run`) | 2–4 min |
| 3b | Fault injection: do Soup's checks catch deliberately broken inputs? (optional) | 5–8 min |
| 4 | Training with `soup train` (LoRA r=16, fp16, 2 epochs × 1,500 rows) | 20–30 min |
| 5 | Merge (`soup merge`) and 4-bit NF4 (`soup merge --save-format 4bit`) | 6–10 min |
| 6 | Negative control: same pipeline, lr ≈ 0 | 5–8 min |
| 7 | Verification `verify_trained.py` (base / adapter / merged / 4-bit / control) + NC2 | 10–15 min |
| 8 | `soup ship` (optional) | 15–25 min |
| 9 | Silent-failure static checks, memory estimate vs measured, numbers for the report, zip | ~2 min |

Total ≈ 75–95 min. Long steps run **in the background** and the cell only waits `MAX_WAIT_S` seconds (env `KIT_MAX_WAIT_S`,
default 1500); if a cell returns "still running", run it again: it resumes waiting and never starts a second copy.

**Rules this notebook follows:** the kernel never touches CUDA (every GPU step is a subprocess with its own peak-memory record,
so nvidia-smi numbers can be attributed); every cell can be re-run (finished steps are skipped unless forced); no number in the
report is typed by hand: `outputs/report_numbers.md` is generated from the run's files. Failed steps are kept, not hidden.

### Why Soup is the primary stack, and what the fallback is for
* The task is posed by the Soup team, and part 3 asks whether **Soup's own checks** (data doctor, pre-flight, `soup ship`) catch
  silent failures. That can only be answered by running the real pipeline through Soup: `soup train` → `soup merge` →
  `soup merge --save-format 4bit` → `soup ship`.
* Soup wraps transformers + PEFT + TRL, so the artifacts are standard HF/PEFT files and can be checked independently of Soup.
  `verify_trained.py` does not import Soup.
* **Fallback** (clearly labelled cells, run only if a Soup step fails): the same hyper-parameters with plain
  transformers + peft + bitsandbytes (`kit_common.py fallback-*`). If it is used, the report must say which steps fell back and why.

### Choices
* **Model `Qwen/Qwen2.5-1.5B-Instruct`** (Apache-2.0): ≤3B, strong multilingual base with a clean ChatML template, fits a T4 in
  fp16 with room for activations. The checkpoint is **bf16**; T4 (sm_75) has no bf16 hardware, so it is trained in **fp16**
  (bf16→fp16 conversion and fp16 overflow are checked, see E2/B3).
* **Dataset `d0rj/alpaca-cleaned-ru`** (CC-BY-4.0, 51,760 rows, columns `instruction/input/output`, plain parquet). Chosen over
  `IlyaGusev/ru_turbo_alpaca` / `ru_turbo_saiga`, which ship only a loading script + `.jsonl.zst` that current `datasets` no longer
  executes. Caveats: machine-translated from alpaca-cleaned (originally text-davinci-003 outputs), so translation artefacts exist,
  and the upstream licence chain includes OpenAI-generated text. Russian is a language the candidate evaluates natively.
* **Sample**: fixed seed 42; filter (empty, code blocks, <60 % Cyrillic answers), exact + near-duplicate removal, 200 held-out eval
  rows first, then 1,500 train rows with no near-duplicate (word-3-gram Jaccard ≥ 0.6) of any eval row.
* **LoRA** r=16, α=32 (scale 2.0), dropout 0.05, all 7 linear projections, lr 2e-4 cosine, warm-up 3 %, 2 epochs,
  micro-batch 2 × grad-accum 8 (effective 16), max_length 1024 (rows longer are dropped, never truncated), gradient checkpointing.
  Micro-batch 2 (not 4) because the memory budget (§2) shows logits dominate: 152k vocab × tokens × ~14 B.

## 0. Setup (≈4–7 min on first run)
Installs `soup-cli[train]` only if missing, checks bitsandbytes and the torchao/peft conflict that Soup's own T4 notebook
documents, and prints versions **in a subprocess** (so this kernel never initialises CUDA).

In [ ]:
import os, sys, subprocess, shutil, importlib.util, json
WORK = "/content/soup-task"
REPO_URL = os.environ.get("SOUP_TASK_REPO", "")   # optional git URL of this kit; leave empty if already cloned/uploaded
FORCE_INSTALL = False
if not os.path.exists(f"{WORK}/kit_common.py"):
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, WORK], check=True)
    else:
        raise SystemExit(f"kit files not found in {WORK}: clone the repo there or upload the files (README.md)")
os.chdir(WORK); sys.path.insert(0, WORK); os.makedirs("logs", exist_ok=True)

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", *args], capture_output=True, text=True)
    with open("logs/pip_install.log", "a") as f:
        f.write(f"\n$ pip {' '.join(args)}\n{r.stdout}\n{r.stderr}\n")
    print(f"pip {' '.join(args)} -> rc={r.returncode}")
    return r.returncode

if FORCE_INSTALL or shutil.which("soup") is None or importlib.util.find_spec("soup_cli") is None:
    pip("install", "-q", "soup-cli[train]")
if importlib.util.find_spec("bitsandbytes") is None:
    pip("install", "-q", "bitsandbytes")
PROBE = "import peft, peft.import_utils as u; f = getattr(u, 'is_torchao_available', None); print('peft', peft.__version__, 'torchao_available', f() if f else 'n/a')"
chk = subprocess.run([sys.executable, "-c", PROBE], capture_output=True, text=True)
print(chk.stdout.strip() or chk.stderr.strip()[-800:])
if chk.returncode != 0 and "ImportError" in chk.stderr and "torchao" in chk.stderr:
    print("peft refuses the preinstalled torchao (ImportError) -> uninstalling it"); pip("uninstall", "-y", "torchao")

VERSIONS = r"""
import sys, json, importlib
out = {"python": sys.version.split()[0]}
for m in ["torch", "transformers", "peft", "trl", "accelerate", "datasets", "bitsandbytes", "safetensors", "soup_cli", "numpy"]:
    try:
        out[m] = getattr(importlib.import_module(m), "__version__", "?")
    except Exception as e:
        out[m] = "MISSING: " + repr(e)[:150]
import torch
out["cuda_available"] = torch.cuda.is_available()
if torch.cuda.is_available():
    out.update(gpu=torch.cuda.get_device_name(0), capability=list(torch.cuda.get_device_capability(0)),
               total_mem_gib=torch.cuda.get_device_properties(0).total_memory / 2**30, torch_cuda=torch.version.cuda,
               bf16_supported_incl_emulation=torch.cuda.is_bf16_supported())
    try:
        out["bf16_in_hardware"] = torch.cuda.is_bf16_supported(including_emulation=False)
    except TypeError:
        out["bf16_in_hardware"] = "n/a in this torch"
print(json.dumps(out))
"""
ver = subprocess.run([sys.executable, "-c", VERSIONS], capture_output=True, text=True)
print(ver.stdout or ver.stderr[-2000:])
open("logs/versions.json", "w").write(ver.stdout)
subprocess.run(f"{sys.executable} -m pip freeze > logs/pip_freeze.txt 2>&1", shell=True)
subprocess.run("nvidia-smi > logs/nvidia_smi_initial.txt 2>&1; free -m > logs/ram_initial.txt; df -h /content > logs/disk_initial.txt; "
               "nproc > logs/cpu_count.txt", shell=True)
print(open("logs/nvidia_smi_initial.txt").read())

### 0.1 Logging and the nvidia-smi sampler (≈5 s)
* `logs/run.log`: Python logging, every stage START/END with millisecond timestamps.
* `logs/nvidia_smi.log`: **raw** `nvidia-smi --query-gpu=timestamp,…,memory.used,… --format=csv -l 1` for the whole session.
  1 s instead of 5 s because the loss/logits spike lasts well under 5 s; it can still miss sub-second peaks, which is why every GPU
  subprocess also records `torch.cuda.max_memory_allocated/reserved` at exit (`logs/peak_*.json`).
* `logs/nvidia_smi_snapshots.log`: full `nvidia-smi` + per-process memory at the start and end of every stage.
* `logs/cmd/<step>.log`: every command's output, one timestamp per line; `logs/stage_memory.jsonl`: per-stage memory summary.

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

K.start_gpu_sampler(interval_s=K.NVSMI_INTERVAL_S)
K.nvsmi_snapshot("session start")
K.log.info("logging ready")

### 0.2 What does the installed Soup actually offer? (≈1–2 min)
The local docs describe v0.75; PyPI may be newer. Every subcommand's `--help` is saved to `logs/soup_help/` and the notebook
adapts (e.g. `soup merge --adapter X` vs `soup merge X`, whether `soup data doctor` exists).

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

with K.stage("soup_cli_discovery"):
    caps = K.soup_caps(refresh=True)
print(json.dumps(caps, indent=1))
for n in ["help_root", "help_train", "help_merge", "help_data", "help_ship", "help_profile"]:
    print(f"\n===== soup {n[5:].replace('_', ' ')} --help =====\n" + K.cmd_output(n)[:3500])

## 1. Data (≈2–4 min)
Download (revision pinned to the dataset commit sha), filter, exact + near-duplicate removal, **held-out first**, then train rows that
are not near-duplicates of any held-out row; rows longer than 1,024 tokens (rendered with the real chat template) are dropped,
never truncated. Writes `data/train.jsonl` (Soup `chatml` format: only `messages`), `data/eval.jsonl`, `data/nc_train.jsonl`
(64 rows for the negative control), `data/probe_prompts.jsonl`, `data/ship_tasks.jsonl`, `data/data_report.json`.

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

with K.stage("data_prep"):
    rep = K.prepare_data(force=False)
show = ["dataset", "dataset_revision", "load_method", "n_raw", "n_pool_after_filters", "n_train", "n_eval", "drop_reasons",
        "leakage_audit", "train_vs_eval_max_jaccard", "default_system_prompt_injected"]
print(json.dumps({k: rep.get(k) for k in show}, indent=1, ensure_ascii=False))
print(json.dumps(rep["token_stats"], indent=1))

### 1.1 Chat template and loss mask (≈30 s, CPU)
Train and inference must render conversations identically, the answer must end with `<|im_end|>` **inside** the trained span (or the
model never learns to stop), and user/system tokens must not be trained. This cell uses the same `encode_example()` that
`verify_trained.py` evaluates with, and, if this Soup version exposes it, Soup's own mask builder
(`soup_cli.data.loss_mask.build_assistant_only_labels`) to compare exactly which tokens Soup trains.

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

with K.stage("template_mask_check"):
    tc = K.masking_check(n=50)
print(json.dumps({k: v for k, v in tc.items() if k != "example_render"}, indent=1, ensure_ascii=False))
print("\n--- example render (answer span that our eval scores is marked) ---\n" + tc["example_render"])

### 1.2 Soup configs
`soup.yaml` (main run), `soup_min.yaml` (only the essential keys, used if this Soup version rejects a key: since v0.75 unknown keys
fail the load), `soup_nc.yaml` (negative control). `quantization: none` is explicit: Soup's default is `4bit` (QLoRA), which would
silently change the method.

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

K.write_soup_config(K.WORK / "soup.yaml", K.soup_config_dict(K.HP, K.DATA / "train.jsonl", K.SOUP_RUN), header="main LoRA run")
K.write_soup_config(K.WORK / "soup_min.yaml", K.soup_config_dict(K.HP, K.DATA / "train.jsonl", K.SOUP_RUN, minimal=True),
                    header="minimal config: used only if soup.yaml is rejected by the dry-run")
K.write_soup_config(K.WORK / "soup_nc.yaml", K.soup_config_dict(K.NC_HP, K.DATA / "nc_train.jsonl", K.NC_RUN),
                    header="negative control: identical pipeline, lr ~ 0 (Soup requires lr > 0), 64 rows, 1 epoch")
for f in ["soup.yaml", "soup_nc.yaml"]:
    print(f"----- {f}\n" + open(f).read())

## 2. Memory budget BEFORE training (≈1 min)
`memory_budget.py` computes the budget from the model's `config.json` and the run parameters: fp16 base weights, LoRA parameters
per module type, fp32 adapter + grads + AdamW states, activations with gradient checkpointing, **logits** (B × T × 151,936 × bytes
incl. fp32 upcast for the loss), allocator + CUDA context, merge and 4-bit quantization peaks. Formulas and assumptions are in its
docstring and printed next to the numbers. Three length scenarios: config ceiling (1024), the longest row in the data (sets the
**peak**, because dynamic padding pads its batch partner up to it), the median row.

Also recorded: Soup's own estimate (`soup profile`) and a CUDA-context calibration (a process holding 4 MiB of tensors: whatever
nvidia-smi shows beyond that is memory torch cannot see).

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

with K.stage("memory_budget"):
    ctx = K.measure_cuda_context()
    ctx_gib = ctx.get("context_overhead_gib")
    args = [sys.executable, "memory_budget.py", "--model", K.MODEL_ID, "--batch-size", str(K.HP["batch_size"]),
            "--grad-accum", str(K.HP["grad_accum"]), "--seq-len", str(K.MAX_LEN), "--data-report", "data/data_report.json",
            "--lora-r", str(K.HP["lora_r"]), "--lora-alpha", str(K.HP["lora_alpha"]), "--lora-dropout", str(K.HP["lora_dropout"]),
            "--target-modules", ",".join(K.TARGET_MODULES), "--out", "outputs/memory_budget.json"]
    if isinstance(ctx_gib, (int, float)) and ctx_gib > 0:
        args += ["--cuda-context-gib", f"{ctx_gib:.3f}"]   # measured, replaces the 0.40 GiB assumption
    rc, out = K.run_cmd("memory_budget", args, timeout_s=300, quiet=True)
    caps = K.soup_caps()
    rc_p, out_p = K.run_cmd("soup_profile", ["soup", "profile", "--config", "soup.yaml"], timeout_s=300, quiet=True)
    if caps.get("profile_has_json"):
        rc_j, out_j = K.run_cmd("soup_profile_json", ["soup", "profile", "--config", "soup.yaml", "--json"], timeout_s=300, quiet=True)
        try:
            K.write_json(K.OUT / "soup_profile.json", json.loads(out_j[out_j.index("{"): out_j.rindex("}") + 1]))
        except Exception as e:
            print("could not parse soup profile --json:", repr(e))
print("CUDA context calibration:", json.dumps(ctx))
print(out)
print("\n===== soup profile (Soup's own estimate) =====\n" + out_p[-4000:])

## 3. Soup's built-in checks: "data doctor" and pre-flight (≈2–4 min)
* **Data doctor**: `soup data validate` / `stats` / `inspect`, plus `soup data doctor --model … --max-length 1024` and its
  `--show-mask` X-ray **if this version has it** (the v0.75 docs list 8 checks: chat_template, template_render, generation_markers,
  eos_in_labels, bos_duplication, system_role, unknown_roles, truncation_risk).
* **Pre-flight**: `soup doctor` (environment, `--config` lists settings the backend will not read), `soup plan` and
  `soup train --dry-run` (config + data validation; in v0.75 the VRAM gate prints only when a run does **not** fit).

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

caps = K.soup_caps()
res = {}
with K.stage("soup_data_checks"):
    for name, args in [("soup_data_validate", ["soup", "data", "validate", "data/train.jsonl"]),
                       ("soup_data_validate_eval", ["soup", "data", "validate", "data/eval.jsonl"]),
                       ("soup_data_stats", ["soup", "data", "stats", "data/train.jsonl"]),
                       ("soup_data_inspect", ["soup", "data", "inspect", "data/train.jsonl"])]:
        res[name], out = K.run_cmd(name, args, timeout_s=600, quiet=True)
        print(f"\n===== {' '.join(args)}  (rc={res[name]}) =====\n" + out[-3000:])
    if caps.get("data_has_doctor"):
        dh = K.cmd_output("help_data_doctor")
        base_args = ["soup", "data", "doctor", "data/train.jsonl", "--model", K.MODEL_ID, "--max-length", str(K.MAX_LEN)]
        args = base_args + (["--output", "outputs/soup_data_doctor.json"] if "--output" in dh else [])
        res["soup_data_doctor"], out = K.run_cmd("soup_data_doctor", args, timeout_s=900, quiet=True)
        print(f"\n===== soup data doctor (rc={res['soup_data_doctor']}) =====\n" + out[-4000:])
        if "--show-mask" in dh:
            rc, out = K.run_cmd("soup_data_doctor_mask", base_args + ["--show-mask", "2"], timeout_s=900, quiet=True)
            print("\n===== soup data doctor --show-mask 2 =====\n" + out[-4000:])
    else:
        print("\n`soup data doctor` does not exist in this Soup version: covered by validate + stats + our §1.1 mask check.")
K.write_json(K.OUT / "soup_data_checks.json", res)

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

caps = K.soup_caps()
pre = {}
with K.stage("soup_preflight"):
    doc = ["soup", "doctor"] + (["--config", "soup.yaml"] if caps.get("doctor_has_config") else [])
    pre["soup_doctor"], out = K.run_cmd("soup_doctor", doc, timeout_s=600, quiet=True)
    print(f"===== {' '.join(doc)} (rc={pre['soup_doctor']}) =====\n" + out[-3500:])
    if caps.get("has_plan"):
        pre["soup_plan"], out = K.run_cmd("soup_plan", ["soup", "plan", "--config", "soup.yaml"], timeout_s=600, quiet=True)
        print(f"\n===== soup plan (rc={pre['soup_plan']}) =====\n" + out[-3500:])
    train_cfg = "soup.yaml"
    if caps.get("train_has_dry_run"):
        pre["dry_run"], out = K.run_cmd("soup_dry_run", ["soup"] + K.soup_train_args("soup.yaml", dry_run=True), timeout_s=900, quiet=True)
        print(f"\n===== soup train --dry-run (rc={pre['dry_run']}) =====\n" + out[-5000:])
        if pre["dry_run"] != 0:
            pre["dry_run_min"], out = K.run_cmd("soup_dry_run_min", ["soup"] + K.soup_train_args("soup_min.yaml", dry_run=True),
                                                timeout_s=900, quiet=True)
            print(f"\n===== minimal config dry-run (rc={pre['dry_run_min']}) =====\n" + out[-4000:])
            train_cfg = "soup_min.yaml" if pre["dry_run_min"] == 0 else None
    K.set_path("train_config", train_cfg)
K.write_json(K.OUT / "soup_preflight.json", pre)
print("\nconfig used for training:", train_cfg, "(None = both rejected: read the log, fix, or use the fallback in §4)")

### 3b. Fault injection: do Soup's checks notice broken inputs? (optional, ≈5–8 min, CPU)
Builds small deliberately broken datasets/configs (25 % duplicates, 50 held-out rows leaked into train, answers far beyond
`max_length`, empty answers, wrong role names, English answers; config with `quantization` omitted, `target_modules: auto`, `lr: 0`,
a typo key, `max_length: 128`, Soup's default lr/r/alpha) and runs `soup data validate/stats/doctor` and `soup train --dry-run` on each.
The table lists exit codes and keywords; **whether Soup "caught" a fault must be judged by reading the logs**, the keyword list is
only a pointer.

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

RUN_FAULTS = True
if RUN_FAULTS:
    with K.stage("fault_injection"):
        rows = K.fault_injection(timeout_s=300)
    print(open(K.OUT / "fault_injection.md").read())

## 4. Training with `soup train` (≈20–30 min, background)
`soup train` runs as a subprocess through the `peak` wrapper (records its CUDA peak + RSS at exit in `logs/peak_soup_train.json`).
Log: `logs/cmd/soup_train.log`. Re-running this cell only waits; `FORCE_RETRAIN=True` starts over.

**Why a falling loss proves nothing here:** it is measured on training batches (memorisation looks the same); it can fall from
learning constant template/header tokens; the saved/merged artefact may not contain what was trained (wrong checkpoint, key mismatch,
fp16 rounding, quantization); fp16 GradScaler can skip steps while loss is still logged. §7 tests each of these directly.

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

FORCE_RETRAIN = False
cfg = K.paths().get("train_config") or "soup.yaml"
if K.find_adapter(K.SOUP_RUN) is not None and K.bg_status("soup_train")[0] in ("done", "absent") and not FORCE_RETRAIN:
    print("adapter already present:", K.find_adapter(K.SOUP_RUN))
else:
    K.run_bg("soup_train", K.peak_cmd("logs/peak_soup_train.json", soup_args=K.soup_train_args(cfg)),
             restart=FORCE_RETRAIN, timeout_s=3 * 3600, peak_json="logs/peak_soup_train.json")
rc = K.wait_bg("soup_train", max_wait_s=MAX_WAIT_S)
print("rc =", rc, "(None = still running -> re-run this cell)")

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

st, rc = K.bg_status("soup_train")
ad = K.find_adapter(K.SOUP_RUN)
if ad is not None and rc == 0:
    K.set_path("adapter", ad); K.set_path("train_root", K.SOUP_RUN); K.set_path("train_backend", "soup")
    K.set_path("train_cmd_name", "soup_train")
else:
    print(f"Soup training produced no adapter (status={st}, rc={rc}). Read logs/cmd/soup_train.log, then run the FALLBACK cell.")
root = K.paths().get("train_root") or K.SOUP_RUN
h = K.training_history(root); s = K.summarize_history(h)
K.write_json(K.OUT / "train_history.json", h); K.write_json(K.OUT / "train_summary.json", s)
png = K.plot_history(h, K.OUT / "loss_curve.png")
print(json.dumps(s, indent=1))
print("peak:", json.dumps(K.read_json(K.LOGS / "peak_soup_train.json"), indent=1))
print("\n--- lines of interest from the Soup training log ---\n" + "\n".join(K.grep_log(K.paths().get("train_cmd_name") or "soup_train")[:80]))
if png:
    from IPython.display import Image, display; display(Image(str(png)))

#### FALLBACK (only if Soup training failed): transformers + peft, same hyper-parameters
Own loop: fp16 autocast + GradScaler (skipped steps are counted), loss normalised over the whole accumulation window, labels only on
answer tokens incl. `<|im_end|>` (same `encode_example` as the evaluation). Runs automatically only when no Soup adapter exists.

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

USE_FALLBACK = K.paths().get("adapter") is None
if USE_FALLBACK:
    a = K.peak_cmd("logs/peak_fallback_train.json", script=K.WORK / "kit_common.py",
                   script_args=["fallback-train", "--train", "data/train.jsonl", "--out", K.rel(K.FALLBACK_RUN), "--hp", json.dumps(K.HP)])
    K.run_bg("fallback_train", a, timeout_s=3 * 3600, peak_json="logs/peak_fallback_train.json")
    rc = K.wait_bg("fallback_train", max_wait_s=MAX_WAIT_S)
    if rc == 0 and K.find_adapter(K.FALLBACK_RUN):
        K.set_path("adapter", K.find_adapter(K.FALLBACK_RUN)); K.set_path("train_root", K.FALLBACK_RUN)
        K.set_path("train_backend", "fallback: transformers+peft"); K.set_path("train_cmd_name", "fallback_train")
        h = K.training_history(K.FALLBACK_RUN); K.write_json(K.OUT / "train_history.json", h)
        K.write_json(K.OUT / "train_summary.json", K.summarize_history(h)); K.plot_history(h, K.OUT / "loss_curve.png")
        print(json.dumps(K.summarize_history(h), indent=1))
    else:
        print("rc =", rc, "(None = still running -> re-run this cell)")
else:
    print("Soup adapter present -> fallback not needed:", K.paths().get("adapter"))

## 5. Merge and 4-bit quantization (≈6–10 min)
* `soup merge` → `outputs/merged` (fp16). In the v0.75 source the merge loads the base with `device_map="cpu"`, so its VRAM peak
  should be ≈0 and its cost is host RAM: compare with the budget.
* `soup merge --save-format 4bit` → `outputs/merged_4bit`: Soup's bitsandbytes-NF4 path (CPU merge → fp16 staged on disk → reload
  with a `BitsAndBytesConfig` on the GPU → save). `soup quantize` only offers gguf/onnx/tensorrt; a GGUF export needs a llama.cpp
  build and its own evaluation runtime, so it is out of scope here and said so in the report.

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

ad = K.paths().get("adapter"); assert ad, "no adapter: finish §4 first"
if not ((K.MERGED / "config.json").exists() and any(K.MERGED.glob("*.safetensors"))):
    K.run_bg("soup_merge", K.peak_cmd("logs/peak_soup_merge.json", soup_args=K.soup_merge_args(ad, K.MERGED)),
             restart=K.bg_status("soup_merge")[0] == "done", timeout_s=3600, peak_json="logs/peak_soup_merge.json")
    rc = K.wait_bg("soup_merge", max_wait_s=MAX_WAIT_S)
    print("rc =", rc)
ok = (K.MERGED / "config.json").exists() and any(K.MERGED.glob("*.safetensors"))
K.set_path("merged", K.MERGED if ok else None)
print("merged ok:", ok, "| peak:", K.read_json(K.LOGS / "peak_soup_merge.json"))
print("files:", sorted(p.name for p in K.MERGED.iterdir()) if K.MERGED.exists() else None)

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

# FALLBACK merge (only if soup merge failed): PEFT merge_and_unload on CPU in fp16, same as Soup's code path
if K.paths().get("merged") is None:
    a = K.peak_cmd("logs/peak_fallback_merge.json", script=K.WORK / "kit_common.py",
                   script_args=["fallback-merge", "--adapter", K.paths()["adapter"], "--out", K.rel(K.MERGED), "--device", "cpu"])
    K.run_bg("fallback_merge", a, timeout_s=3600, peak_json="logs/peak_fallback_merge.json")
    rc = K.wait_bg("fallback_merge", max_wait_s=MAX_WAIT_S)
    if rc == 0:
        K.set_path("merged", K.MERGED); K.set_path("merge_backend", "fallback")
else:
    print("merged present -> fallback not needed")

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

caps = K.soup_caps(); ad = K.paths().get("adapter")
if not (K.MERGED_4BIT / "config.json").exists() and caps.get("merge_has_save_format"):
    K.run_bg("soup_quantize_4bit", K.peak_cmd("logs/peak_soup_quantize_4bit.json",
             soup_args=K.soup_merge_args(ad, K.MERGED_4BIT, ["--save-format", "4bit"])),
             restart=K.bg_status("soup_quantize_4bit")[0] == "done", timeout_s=3600, peak_json="logs/peak_soup_quantize_4bit.json")
    rc = K.wait_bg("soup_quantize_4bit", max_wait_s=MAX_WAIT_S)
    print("rc =", rc)
qc = (K.read_json(K.MERGED_4BIT / "config.json", {}) or {}).get("quantization_config")
K.set_path("merged_4bit", K.MERGED_4BIT if qc else None)
print("4-bit quantization_config:", json.dumps(qc, indent=1))
print("peak:", K.read_json(K.LOGS / "peak_soup_quantize_4bit.json"))

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

# FALLBACK 4-bit (only if Soup's 4-bit path failed): BitsAndBytesConfig nf4 + double quant, fp16 compute, from the merged fp16
if K.paths().get("merged_4bit") is None and K.paths().get("merged"):
    a = K.peak_cmd("logs/peak_fallback_quantize.json", script=K.WORK / "kit_common.py",
                   script_args=["fallback-quantize", "--merged", K.paths()["merged"], "--out", K.rel(K.MERGED_4BIT)])
    K.run_bg("fallback_quantize", a, timeout_s=3600, peak_json="logs/peak_fallback_quantize.json")
    rc = K.wait_bg("fallback_quantize", max_wait_s=MAX_WAIT_S)
    if rc == 0:
        K.set_path("merged_4bit", K.MERGED_4BIT); K.set_path("quant_backend", "fallback")
else:
    print("4-bit present (or no merged model) -> fallback not run")

## 6. Negative control (≈5–8 min)
Same pipeline (Soup train → Soup merge), but **lr = 1e-9** on 64 rows for 1 epoch: a run that "finishes successfully" without
meaningfully changing the model. Soup's schema rejects `lr: 0` (`lr > 0`): §3b's `cfg_lr_zero` dry-run records that. The verification
in §7 must flag this control as **untrained**; if it does not, the check is worthless.

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

backend = K.paths().get("train_backend") or "soup"
if K.find_adapter(K.NC_RUN) is None:
    if backend == "soup":
        K.run_bg("soup_nc_train", K.peak_cmd("logs/peak_soup_nc_train.json", soup_args=K.soup_train_args("soup_nc.yaml")),
                 restart=K.bg_status("soup_nc_train")[0] == "done", timeout_s=3600, peak_json="logs/peak_soup_nc_train.json")
        rc = K.wait_bg("soup_nc_train", max_wait_s=MAX_WAIT_S)
    else:
        a = K.peak_cmd("logs/peak_fallback_nc_train.json", script=K.WORK / "kit_common.py",
                       script_args=["fallback-train", "--train", "data/nc_train.jsonl", "--out", K.rel(K.NC_RUN), "--hp", json.dumps(K.NC_HP)])
        K.run_bg("fallback_nc_train", a, timeout_s=3600, peak_json="logs/peak_fallback_nc_train.json")
        rc = K.wait_bg("fallback_nc_train", max_wait_s=MAX_WAIT_S)
nc = K.find_adapter(K.NC_RUN)
K.set_path("nc_adapter", nc)
if nc is not None and not (K.NC_MERGED / "config.json").exists():
    if (K.paths().get("merge_backend") or "soup") == "soup":
        K.run_bg("soup_nc_merge", K.peak_cmd("logs/peak_soup_nc_merge.json", soup_args=K.soup_merge_args(nc, K.NC_MERGED)),
                 restart=K.bg_status("soup_nc_merge")[0] == "done", timeout_s=3600, peak_json="logs/peak_soup_nc_merge.json")
        K.wait_bg("soup_nc_merge", max_wait_s=MAX_WAIT_S)
    else:
        a = K.peak_cmd("logs/peak_fallback_nc_merge.json", script=K.WORK / "kit_common.py",
                       script_args=["fallback-merge", "--adapter", str(nc), "--out", K.rel(K.NC_MERGED), "--device", "cpu"])
        K.run_bg("fallback_nc_merge", a, timeout_s=3600); K.wait_bg("fallback_nc_merge", max_wait_s=MAX_WAIT_S)
K.set_path("nc_merged", K.NC_MERGED if (K.NC_MERGED / "config.json").exists() else None)
h = K.training_history(K.NC_RUN); K.write_json(K.OUT / "nc_train_summary.json", K.summarize_history(h))
print("NC adapter:", nc, "| NC merged:", K.paths().get("nc_merged"))
print("NC training summary (it 'finished fine'):", json.dumps(K.summarize_history(h), indent=1))

## 7. Verification: is the model real? (≈10–15 min)
`verify_trained.py` (standalone, does not import Soup) loads **base, base+adapter (unmerged), base+control, merged fp16, the 4-bit
artifact and the base quantized with the same 4-bit config** and evaluates them in lockstep on the same 200 held-out rows with the
same chat template:

* **A** adapter file read directly: `‖lora_B‖` (B is zero at init), modules found vs 28 × 7 expected, every module maps onto a base
  weight, file tensors == what PEFT loaded, predicted fp16 rounding loss of the update.
* **B** merged weights == fp16(base + α/r · B·A) computed independently; non-LoRA parameters bit-identical; relative ΔW per module type.
* **C** held-out NLL on answer tokens only, paired bootstrap 95 % CI; 4-bit gain measured against the **4-bit base** (like with like).
* **D** KL + top-1 agreement of next-token distributions: merged↔base (did it change?), adapter↔merged (did the merge preserve it?),
  merged↔4-bit (quant fidelity), base↔base-4-bit (quantization noise floor to compare the training effect against).
* **Q** 4-bit codes: artifact == quantize(merged) and ≠ quantize(base). **E** greedy generations side by side, EOS, language,
  repetition, inf/NaN logits. **T** tokenizer/template/generation_config consistency.

Exit code 2 means "verdict: not trained", not a crash.

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

P = K.paths()
assert P.get("adapter") and P.get("merged"), "need adapter + merged (sections 4-5)"
args = ["--base", K.MODEL_ID, "--adapter", P["adapter"], "--merged", P["merged"], "--eval-data", "data/eval.jsonl",
        "--train-data", "data/train.jsonl", "--prompts", "data/probe_prompts.jsonl", "--out-dir", K.rel(K.VERIFY_DIR),
        "--max-length", str(K.MAX_LEN)]
if P.get("merged_4bit"): args += ["--quant4", P["merged_4bit"]]
if P.get("nc_adapter"): args += ["--control-adapter", P["nc_adapter"]]
if P.get("nc_merged"): args += ["--control-merged", P["nc_merged"]]
K.run_bg("verify", K.peak_cmd("logs/peak_verify.json", script=K.WORK / "verify_trained.py", script_args=args),
         timeout_s=2 * 3600, peak_json="logs/peak_verify.json")
rc = K.wait_bg("verify", max_wait_s=MAX_WAIT_S)
print("rc =", rc, "(0 = trained, 2 = NOT trained, None = still running)")
if (K.VERIFY_DIR / "verify_summary.txt").exists():
    print(open(K.VERIFY_DIR / "verify_summary.txt").read())

### 7.1 Second negative control (NC2): "the adapter is fine but the merge did not apply it" (≈4 min)
Runs the same script with `--merged` pointing at the **base model**. Expected: A-checks PASS (the adapter is real) while B1/B2/C1/D1
FAIL. This demonstrates that the merge checks catch a merge that silently produced the base.

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

P = K.paths()
args = ["--base", K.MODEL_ID, "--adapter", P["adapter"], "--merged", K.MODEL_ID, "--eval-data", "data/eval.jsonl",
        "--out-dir", K.rel(K.VERIFY_NC2_DIR), "--max-eval", "100", "--skip-generations", "--max-length", str(K.MAX_LEN)]
K.run_bg("verify_nc2", K.peak_cmd("logs/peak_verify_nc2.json", script=K.WORK / "verify_trained.py", script_args=args), timeout_s=3600)
rc = K.wait_bg("verify_nc2", max_wait_s=MAX_WAIT_S)
if (K.VERIFY_NC2_DIR / "verify_summary.txt").exists():
    print(open(K.VERIFY_NC2_DIR / "verify_summary.txt").read())

## 8. `soup ship` (optional, ≈15–25 min)
Live run: base vs merged fp16 with a Russian task-eval (`data/ship_tasks.jsonl`: 30 held-out prompts scored `semantic` against the
reference, 5 Russian facts `contains`, 5 "answer is Russian" `regex`) plus a subset of Soup's bundled leg-2 suites (to fit the T4
time budget; `SHIP_SUITES=""` uses Soup's default). Watch the log for the precision `soup ship` loads models in (v0.75 and
v0.77.2 source: bf16 on CUDA without `--config`; Qwen's shipped `repetition_penalty=1.1` still applies to its greedy decode) and note that it evaluates **merged fp16, not the 4-bit artifact we would ship**.
The second cell feeds **our** held-out metric (geometric-mean token probability `exp(−NLL)`, base fp16 vs 4-bit artifact) plus the
live leg-2 scores into `soup ship --evidence`, to see Soup's decision rule on a metric that actually moves.

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

RUN_SHIP = True
SHIP_SUITES = "mini_mmlu,mini_instruction"
caps = K.soup_caps(); P = K.paths()
if RUN_SHIP and P.get("merged"):
    args = ["ship", "--base", K.MODEL_ID, "--tuned", P["merged"], "--task-eval", "data/ship_tasks.jsonl",
            "--output", "outputs/ship_verdict.json", "--emit-evidence", "outputs/ship_evidence.json"]
    if caps.get("ship_has_device"): args += ["--device", "cuda"]
    if SHIP_SUITES and caps.get("ship_has_general_suite"): args += ["--general-suite", SHIP_SUITES]
    K.run_bg("soup_ship", K.peak_cmd("logs/peak_soup_ship.json", soup_args=args), timeout_s=90 * 60, peak_json="logs/peak_soup_ship.json")
    rc = K.wait_bg("soup_ship", max_wait_s=MAX_WAIT_S)
    print("rc =", rc, "(0 SHIP, 2 DON'T SHIP, 3 usage error, 1 runtime error, None still running)")
    if rc == 3 and SHIP_SUITES:
        print("usage error -> retrying with Soup's default leg-2 suite")
        args = [a for a in args if a not in ("--general-suite", SHIP_SUITES)]
        K.run_bg("soup_ship_default", K.peak_cmd("logs/peak_soup_ship_default.json", soup_args=args), timeout_s=90 * 60)
        rc = K.wait_bg("soup_ship_default", max_wait_s=MAX_WAIT_S)
    print(json.dumps(K.read_json(K.OUT / "ship_verdict.json"), indent=1, ensure_ascii=False))

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

V = K.read_json(K.VERIFY_DIR / "verify_results.json", {}) or {}
H = V.get("heldout") or {}
if "base" in H:
    tuned = "merged_4bit" if "merged_4bit" in H else "merged"
    live = K.read_json(K.OUT / "ship_evidence.json", {}) or {}
    ev = {"task": {"mode": "metric", "base": math.exp(-H["base"]["nll_per_token"]), "tuned": math.exp(-H[tuned]["nll_per_token"])},
          "benchmarks": live.get("benchmarks", {})}
    K.write_json(K.OUT / "ship_evidence_custom.json", ev)
    print("evidence (computed from verify_results.json, tuned =", tuned, "):", json.dumps(ev, indent=1))
    rc, out = K.run_cmd("soup_ship_offline", ["soup", "ship", "--evidence", "outputs/ship_evidence_custom.json",
                                              "--output", "outputs/ship_verdict_offline.json"], timeout_s=300, quiet=True)
    print(f"rc={rc}\n" + out[-3000:])
else:
    print("verification results missing: run §7 first")

## 9. Silent-failure checks, memory estimate vs measured, report numbers, bundle (≈2 min)

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

with K.stage("silent_checks"):
    sc = K.static_artifact_checks()
for c in sc["checks"]:
    print(f"[{c['status']:<7}] {c['id']:<24} {c['what']}  ->  {json.dumps(c['value'], ensure_ascii=False, default=str)[:220]}")
print("\ntraining summary:", json.dumps(sc["training_summary"], indent=1))

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

meas = K.collect_measured()
print(json.dumps(meas, indent=1))
rc, out = K.run_cmd("memory_compare", [sys.executable, "memory_budget.py", "--compare", "outputs/measured_memory.json",
                                       "--estimate", "outputs/memory_budget.json", "--compare-out", "outputs/memory_compare"],
                    timeout_s=120, quiet=True)
print(out)

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

p = K.report_numbers()
print(open(p).read())

In [ ]:
import os, sys, json, math, importlib
sys.path.insert(0, "/content/soup-task")
import kit_common as K; importlib.reload(K); K.init()
MAX_WAIT_S = int(os.environ.get("KIT_MAX_WAIT_S", "1500"))   # waiting cells return after this; re-run to keep waiting

K.nvsmi_snapshot("before bundle")
K.stop_gpu_sampler()      # flushes logs/nvidia_smi.log; K.start_gpu_sampler() restarts it if you continue
zp = K.bundle()
print("send this file:", zp)
try:
    from google.colab import files
    files.download(str(zp))
except Exception as e:
    print("download it from the Colab file browser (left panel):", zp, "|", repr(e)[:100])

### What to send back
1. `soup_task_results_<timestamp>.zip` (logs, nvidia-smi raw output, all JSON/MD/PNG outputs, data reports, configs, the kit code).
   Model weights are excluded on purpose.
2. This notebook **with outputs**: File → Download → Download .ipynb.
3. Fill `report_template.md` from `outputs/report_numbers.md`, `outputs/verify/verify_summary.txt`, `outputs/memory_compare.md`,
   `outputs/fault_injection.md`, `outputs/silent_checks.json` and `silent_failures.md`. If a step failed, report the failure.